# Initial Label Generation

Generates BIO-tagged training data using the following method:
- `dslim/bert-large-NER` detects PERSON entities in article sentences
- Detected names are fuzzy-matched against the DB `Player` table to confirm they are footballers
- Keyword proximity scanning within a token window around each confirmed player tags INJURY and STATUS spans

Output: `data/initial_labels.jsonl` — one sentence per line with BIO tags for `B-PLAYER / I-PLAYER / B-INJURY / I-INJURY / B-STATUS / I-STATUS / O`.

Since the initial dataset will be noisy, it will paired with an active learning steps where an LLM will correct ambiguous explanations.

In [8]:
import json
import re
import sqlite3
from pathlib import Path

import pandas as pd
from rapidfuzz import process, fuzz
from transformers import pipeline

REPO_ROOT  = Path.cwd().parent.parent if Path.cwd().name == 'finetune' else Path.cwd()
DB_PATH    = REPO_ROOT / 'data' / 'database.sqlite'
CACHE_PATH = REPO_ROOT / 'data' / 'articles_cache.json'
OUTPUT_PATH = REPO_ROOT / 'data' / 'initial_labels.jsonl'

assert DB_PATH.exists(),    f'Missing {DB_PATH}'
assert CACHE_PATH.exists(), f'Missing {CACHE_PATH}'

print(f'DB:    {DB_PATH}')
print(f'Cache: {CACHE_PATH} ({CACHE_PATH.stat().st_size / 1024**2:.1f} MB)')

DB:    /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/database.sqlite
Cache: /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/articles_cache.json (74.4 MB)


## Load player names from DB

Build a lookup list of all known footballer names from the `Player` table. This is the used to filter out non-footballer PERSON entities detected by the NER model (e.g. managers, commentators, politicians).

In [9]:
conn = sqlite3.connect(DB_PATH)
players_df = pd.read_sql('SELECT player_api_id, player_name FROM Player', conn)
conn.close()

player_names = players_df['player_name'].tolist()

print(f'Loaded {len(player_names):,} player names from DB')
print('Sample:', player_names[:10])

Loaded 11,060 player names from DB
Sample: ['Aaron Appindangoye', 'Aaron Cresswell', 'Aaron Doran', 'Aaron Galindo', 'Aaron Hughes', 'Aaron Hunt', 'Aaron Kuhl', 'Aaron Lennon', 'Aaron Lennox', 'Aaron Meijers']


## Load bert-large-NER

`dslim/bert-large-NER` is fine-tuned on CoNLL-2003 for NER. It tags tokens as PER, ORG, LOC, or MISC. We only use PER entities — candidate player names verified against the DB.

`aggregation_strategy="simple"` merges subword tokens back into full words automatically (e.g. `Öz`, `##il` → `Özil`).

In [10]:
ner_pipeline = pipeline(
    task='ner',
    model='dslim/bert-large-NER',
    aggregation_strategy='simple',
)
print('bert-large-NER loaded.')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-large-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bert-large-NER loaded.


## Fuzzy match helper

For each PERSON entity from the NER model, check if it matches a known player in the DB using `rapidfuzz`. `fuzz.token_set_ratio` handles partial name matches well (e.g. "Kane" matching "Harry Kane"). Threshold set to 85 so that lower catches more players but increases false positives.

In [11]:
from functools import lru_cache

# Pre-build token sets from all player names for O(1) pre-filter
_player_name_tokens = {
    tok.lower()
    for name in player_names
    for tok in name.split()
}

FUZZY_THRESHOLD = 85

@lru_cache(maxsize=16384)
def is_known_player(name: str, threshold: int = FUZZY_THRESHOLD) -> bool:
    """Returns True if name fuzzy-matches a player in the DB above the threshold."""
    # Fast pre-filter: skip fuzzy search if no token overlaps any known player name token
    name_tokens = {t.lower().strip(".,;:!?\"'") for t in name.split()}
    if not name_tokens & _player_name_tokens:
        return False
    match = process.extractOne(name, player_names, scorer=fuzz.token_set_ratio)
    if match is None:
        return False
    return match[1] >= threshold

# Sanity check — players should return True, non-players False
test_names = ['Mesut Ozil', 'Harry Kane', 'Pep Guardiola', 'Boris Johnson']
for name in test_names:
    print(f'  {name:<20} {is_known_player(name)}')


  Mesut Ozil           True
  Harry Kane           True
  Pep Guardiola        False
  Boris Johnson        False


## Keyword lists for INJURY and STATUS

Onece a `person` entity has been matched, look at surrounding context window of 10 look for the injury and status keywords. `INJURY` covers body part / injury type terms. `STATUS` covers availability terms.

In [12]:
INJURY_KEYWORDS = [
    'hamstring', 'knee', 'ankle', 'muscle', 'strain', 'torn', 'fracture',
    'concussion', 'back', 'thigh', 'groin', 'shoulder', 'calf', 'foot',
    'wrist', 'ligament', 'achilles', 'hip', 'rib',
]

STATUS_KEYWORDS = [
    'out', 'doubtful', 'questionable', 'doubt', 'ruled out', 'miss',
    'missing', 'unavailable', 'suspended', 'suspension', 'banned',
    'injured', 'injury', 'fitness', 'sidelined', 'placed on ir',
]

INJURY_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in INJURY_KEYWORDS) + r')\b',
    re.IGNORECASE
)
STATUS_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in STATUS_KEYWORDS) + r')\b',
    re.IGNORECASE
)

PROXIMITY_WINDOW = 10

## BIO tagging function

Takes a single sentence and returns a list of `(token, tag)` pairs:
1. Run bert-large-NER → find PER entities
2. Fuzzy-match each PER against the player DB → keep confirmed footballers only
3. Assign `B-PLAYER` / `I-PLAYER` to player name tokens
4. Scan ±10 tokens around each player for INJURY / STATUS keywords
5. Skip the sentence entirely if no confirmed player is found

In [13]:
def tag_sentence(sentence, ner_results):
    """
    Returns list of (token, BIO_tag) for a sentence, or None if no confirmed player found.
    ner_results: pre-computed output from ner_pipeline for this sentence (list of entity dicts).
    """
    tokens = sentence.split()
    if not tokens:
        return None

    # Filter to confirmed footballer PER entities
    player_spans = []
    for ent in ner_results:
        if ent['entity_group'] == 'PER' and is_known_player(ent['word']):
            player_spans.append((ent['start'], ent['end'], ent['word']))

    if not player_spans:
        return None

    # Default all tokens to O
    tags = ['O'] * len(tokens)

    # Reconstruct character offsets per token
    char_pos = 0
    token_offsets = []
    for tok in tokens:
        start = sentence.find(tok, char_pos)
        end   = start + len(tok)
        token_offsets.append((start, end))
        char_pos = end

    # Tag PLAYER tokens using character offsets from NER
    player_token_indices = set()
    for p_start, p_end, _ in player_spans:
        first = True
        for i, (t_start, t_end) in enumerate(token_offsets):
            if t_start >= p_start and t_end <= p_end:
                tags[i] = 'B-PLAYER' if first else 'I-PLAYER'
                player_token_indices.add(i)
                first = False

    # Scan proximity window around each player token for INJURY / STATUS
    for p_idx in player_token_indices:
        window_start = max(0, p_idx - PROXIMITY_WINDOW)
        window_end   = min(len(tokens), p_idx + PROXIMITY_WINDOW + 1)

        for i in range(window_start, window_end):
            if tags[i] != 'O':
                continue
            tok_clean = tokens[i].strip('.,;:!?"\' ')
            if INJURY_PATTERN.fullmatch(tok_clean):
                tags[i] = 'B-INJURY'
            elif STATUS_PATTERN.fullmatch(tok_clean):
                tags[i] = 'B-STATUS'

    return list(zip(tokens, tags))


# Quick test — pass dummy ner_results matching what the pipeline would return
test_sentence = "Kane is ruled out for three weeks with a hamstring injury."
test_ner = [{'entity_group': 'PER', 'word': 'Kane', 'start': 0, 'end': 4, 'score': 0.99}]
result = tag_sentence(test_sentence, test_ner)
if result:
    for token, tag in result:
        if tag != 'O':
            print(f'  {token:<25} {tag}')
else:
    print('No confirmed player found.')


  Kane                      B-PLAYER
  out                       B-STATUS
  hamstring                 B-INJURY
  injury.                   B-STATUS


## Run over all cached articles and write initial labels

Iterates over every article in `articles_cache.json`, splits each body into sentences with `nltk.sent_tokenize`, runs `tag_sentence()` on each, and writes sentences that contain at least one confirmed player to `data/initial_labels.jsonl`.

Each line is a JSON object:
```json
{"sentence": "...", "tokens": ["Kane", "is", ...], "tags": ["B-PLAYER", "O", ...]}
```

Report progress every 500 articles. Sentences with no confirmed player are silently skipped.

In [14]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

NER_BATCH_SIZE = 32  # sentences per NER forward pass

cache = json.loads(CACHE_PATH.read_text(encoding='utf-8'))
articles = [article for article_list in cache.values() for article in article_list]
print(f'Total cached articles: {len(articles):,}')

written = 0
skipped_no_player = 0

with OUTPUT_PATH.open('w', encoding='utf-8') as fout:
    for idx, article in enumerate(articles):
        body = article.get('body', '') or ''
        if not body.strip():
            continue

        sentences = sent_tokenize(body)
        if not sentences:
            continue

        # Batch NER over all sentences in this article at once
        batch_ner_raw = ner_pipeline(sentences, batch_size=NER_BATCH_SIZE)
        # pipeline returns list[dict] for a single string, list[list[dict]] for a list —
        # normalise to always be list[list[dict]]
        if sentences and isinstance(batch_ner_raw[0], dict):
            batch_ner = [batch_ner_raw]
        else:
            batch_ner = batch_ner_raw

        for sentence, ner_results in zip(sentences, batch_ner):
            result = tag_sentence(sentence, ner_results)
            if result is None:
                skipped_no_player += 1
                continue

            tokens, tags = zip(*result)
            record = {
                'sentence': sentence,
                'tokens':   list(tokens),
                'tags':     list(tags),
            }
            fout.write(json.dumps(record) + '\n')
            written += 1

        if (idx + 1) % 500 == 0:
            print(f'  [{idx+1:>6,} / {len(articles):,}]  written={written:,}  skipped={skipped_no_player:,}')

print(f'\nDone.  Sentences written : {written:,}')
print(f'       Sentences skipped : {skipped_no_player:,}')
print(f'       Output            : {OUTPUT_PATH}')


Total cached articles: 12,123
  [   500 / 12,123]  written=9,309  skipped=18,901
  [ 1,000 / 12,123]  written=14,759  skipped=32,338
  [ 1,500 / 12,123]  written=21,758  skipped=45,539
  [ 2,000 / 12,123]  written=29,477  skipped=59,971
  [ 2,500 / 12,123]  written=36,679  skipped=72,585
  [ 3,000 / 12,123]  written=45,038  skipped=86,782
  [ 3,500 / 12,123]  written=52,527  skipped=98,995
  [ 4,000 / 12,123]  written=64,430  skipped=119,791
  [ 4,500 / 12,123]  written=80,274  skipped=145,226
  [ 5,000 / 12,123]  written=88,887  skipped=160,377
  [ 5,500 / 12,123]  written=97,669  skipped=175,422
  [ 6,000 / 12,123]  written=106,140  skipped=190,038
  [ 7,000 / 12,123]  written=122,694  skipped=216,440
  [ 7,500 / 12,123]  written=132,132  skipped=231,936
  [ 8,000 / 12,123]  written=143,196  skipped=248,426
  [ 8,500 / 12,123]  written=160,109  skipped=276,464
  [ 9,000 / 12,123]  written=169,745  skipped=297,073
  [ 9,500 / 12,123]  written=182,127  skipped=325,279
  [10,000 / 12,12

## Sanity check initial labels

Loads `initial_labels.jsonl` and samples sentences that contain at least one non-O tag, printing only the tagged tokens to make the output readable. Also shows a tag frequency breakdown so you can eyeball label balance.

In [15]:
from collections import Counter
import random

records = []
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

print(f'Total labelled sentences: {len(records):,}\n')

# Tag frequency breakdown
tag_counts = Counter()
for rec in records:
    tag_counts.update(rec['tags'])

print('Tag distribution:')
total_tokens = sum(tag_counts.values())
for tag, count in sorted(tag_counts.items()):
    print(f'  {tag:<12} {count:>8,}  ({100 * count / total_tokens:.2f}%)')

# Sample 10 sentences with at least one INJURY or STATUS tag
interesting = [r for r in records if any(t != 'O' and t != 'B-PLAYER' and t != 'I-PLAYER' for t in r['tags'])]
sample = random.sample(interesting, min(10, len(interesting)))

print(f'\n--- Sample sentences with INJURY / STATUS tags ({len(interesting):,} total) ---\n')
for rec in sample:
    print('SENTENCE:', rec['sentence'])
    tagged = [(tok, tag) for tok, tag in zip(rec['tokens'], rec['tags']) if tag != 'O']
    for tok, tag in tagged:
        print(f'  {tok:<25} {tag}')
    print()

Total labelled sentences: 226,385

Tag distribution:
  B-INJURY       11,479  (0.21%)
  B-PLAYER      286,385  (5.12%)
  B-STATUS       17,284  (0.31%)
  I-PLAYER       99,995  (1.79%)
  O            5,183,170  (92.58%)

--- Sample sentences with INJURY / STATUS tags (24,814 total) ---

SENTENCE: After another one-way exchange and the reporter admitting that he wanted to get "a little rise" out of the player, Schweinsteiger turned to Bayern's media officer.
  out                       B-STATUS
  Schweinsteiger            B-PLAYER

SENTENCE: Portland attack again and Hurtado is forced to concede a corner under pressure... Nagbe back defending and his speed is necessary as Neagle chases a ball over the top down the left.
  Hurtado                   B-PLAYER
  back                      B-INJURY

SENTENCE: Fulham had already sensed Liverpool's weakness when Dirk Kuyt was caught out by Paul Konchesky – "a massive mistake," said Benítez – and the full-back's cross was headed back by Zoltan G